--------------------------------------------------------------------------------------------------------------------

**This code uses the following dataset:**

    - Thredd's UCAR ERA5 Reanalysis data on a .25 Degree Latitude/Longitude Grid:
    https://thredds.rda.ucar.edu/thredds/catalog/files/g/d633000/e5.oper.an.pl/202112/catalog.html
  
--------------------------------------------------------------------------------------------------------------------

## ERA5 Integrated Vapor Transport (IVT) & Mean Sea-Level Pressure (MSLP)

This notebook computes and plots **Integrated Vapor Transport (IVT)** and **Mean Sea-Level Pressure (MSLP)** from locally stored ERA5 GRIB files. IVT is calculated with trapezoidal integration of specific humidity and wind across multiple pressure levels, used for **atmospheric river** identification and moisture flux analysis. (https://cw3e.ucsd.edu/)

### Physical Background

**IVT** measures the vertically integrated flux of water vapor through an atmospheric column:
$$\text{IVT} = \frac{1}{g} \int_{p_{\text{top}}}^{p_{\text{sfc}}} q \cdot \vec{V} \, dp$$

- Where:
    - $q$ = specific humidity (kg/kg)
    - $\vec{V}$ = horizontal wind vector (m/s)
    - $dp$ = pressure layer thickness (Pa)
    - $g$ = 9.81 m/s²

**IVT ≥ 250 kg/m/s is the commonly used threshold for atmospheric river identification.**

**MSLP** contours help illustrate the pressure systems at the surface that drive moisture transport.

In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

# redirect cfgrib index files to home directory
index_dir = Path("/data1/alamia/.cfgrib_indexes")
index_dir.mkdir(exist_ok=True)
os.environ["CFGRIB_INDEX_PATH"] = str(index_dir)

import xarray as xr
import numpy as np
from datetime import datetime as dt
from metpy.units import units
from metpy import calc as mpcalc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cfgrib  # noqa: F401 — imported to register the cfgrib xarray backend

## 1.) ERA5 GRIB Data Loaders

Two loader functions handle reading from the read-only `/data1/ERA5` archive:

- **`get_era5_ds`**: pressure-level variables (U, V, q), filtered by level (hPa)
- **`get_era5_sfc_ds`**: surface-level variables (MSLP), filtered by `typeOfLevel='surface'`

Both follow the same pattern:
1. Locate the GRIB file at the expected path
2. Copy it to a temporary directory cfgrib can write index files into
3. Open with `xr.open_dataset(..., engine='cfgrib')` and force-load into memory
4. Delete the temporary copy in a `finally` block to guarantee cleanup

Chunking along `time` (`chunks={'time': 24}`) enables Dask-lazy loading before the `.load()` call pulls the selected timestep into memory.

In [ ]:
# loads a single pressure-level variable from a local GRIB file because /data1/ERA5 is read-only, each file is copied to a temp directory
# ERA5 local root
era5_root = Path("/data1/ERA5")

def get_era5_ds(variable: str, year: int, level: int):
   
    filepath = era5_root / variable / f"{variable}_{year}.grib"
    if not filepath.exists():
        raise FileNotFoundError(f"No GRIB file found at {filepath}")
    
    # copy to a temp dir you own so cfgrib can write its index file
    tmp_dir = Path(tempfile.mkdtemp(dir="/data1/alamia/.cfgrib_indexes"))
    tmp_file = tmp_dir / filepath.name
    shutil.copy2(filepath, tmp_file)
    
    try:
        ds = xr.open_dataset(
            tmp_file,
            engine='cfgrib',
            filter_by_keys={'typeOfLevel': 'isobaricInhPa', 'level': level},
            chunks={'time': 24}
        )
        ds = ds.load()  # force load into memory so we no longer need the temp file
    finally:
        shutil.rmtree(tmp_dir)  # clean up, even if something errors
    
    return ds

# load a single surface-level ERA5 variable from grib file
def get_era5_sfc_ds(variable: str, year: int):
    filepath = era5_root / "surface" / variable / f"{variable}_{year}.grib"
    
    if not filepath.exists():
        raise FileNotFoundError(f"No GRIB file found at {filepath}")
    
    tmp_dir = Path(tempfile.mkdtemp(dir="/data1/alamia/.cfgrib_indexes"))
    tmp_file = tmp_dir / filepath.name
    shutil.copy2(filepath, tmp_file)
    
    try:
        ds = xr.open_dataset(
            tmp_file,
            engine='cfgrib',
            filter_by_keys={'typeOfLevel': 'surface'},
            chunks={'time': 24}
        )
        ds = ds.load()
    finally:
        shutil.rmtree(tmp_dir)
    
    return ds

## 2.) User Configuration

All run parameters are defined here. No edits should be needed below this cell.

**Extent options:**
- Inner 48 states: `lonW, lonE, latS, latN = -134, -59, 23, 50`
- Including Alaska *(current)*: `lonW, lonE, latS, latN = -180, -30, 0, 70`

**Note:** Coordinates are padded by +-5deg when slicing ERA5 data to avoid edge artifacts in the contour plots near the map boundary.

In [ ]:
# specify date/time
Year, Month, Day, Hour, Minute = 2023, 2, 23, 0, 0
dateTime = dt(Year, Month, Day, Hour, Minute)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")

# data dimensions
# lonW, lonE, latS, latN = -134, -59, 23, 50 # for the inner 48 states
lonW, lonE, latS, latN = -180, -30, 0, 70    # to include alaska

# IVT integration levels (hPa)
ivt_levels = [1000, 925, 850, 700, 600, 500, 400, 300]

# define the folder path
plot_dir = Path("/home1/alamia/Research/ERA5")

## 3.) IVT Calculation (Trapezoidal Integration)

IVT is computed by summing trapezoidal layer contributions between pressure levels. For each pair of levels $(p_a, p_b)$:

$$\text{IVT}_{u} \mathrel{+}= \frac{q_a U_a + q_b U_b}{2} \cdot \frac{\Delta p}{g}$$

$$\text{IVT}_{v} \mathrel{+}= \frac{q_a V_a + q_b V_b}{2} \cdot \frac{\Delta p}{g}$$

- where $\Delta p = (p_a - p_b) \times 100$ Pa. 

A level cache avoids reloading the same GRIB file multiple times across loop iterations, since `U`, `V`, and `q` share the same pressure-level files.

**Note:** This step typically takes a few minutes depending on file size and available memory

In [ ]:
# level cache that avoids GRIB reading over the same (variable, level) 
# which happens for the upper and lower boundary of touching layers
level_cache = {}
def load_level(var, year, level):
    if (var, level) not in level_cache:
        level_cache[(var, level)] = get_era5_ds(var, year, level)
    return level_cache[(var, level)]
    
# initialize IVT components as scalars
IVT_u, IVT_v = 0.0, 0.0

for i in range(len(ivt_levels) - 1):
    lev_a = ivt_levels[i]       # upper boundary of layer
    lev_b = ivt_levels[i + 1]   # lower boundary of layer
    dp = (lev_b - lev_a) * 100  # layer thickness from hPa to Pa

    # load wind and humidity at each level boundary
    U_a = load_level("U", Year, lev_a)['u'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    U_b = load_level("U", Year, lev_b)['u'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    V_a = load_level("V", Year, lev_a)['v'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    V_b = load_level("V", Year, lev_b)['v'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    Q_a = load_level("q", Year, lev_a)['q'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    Q_b = load_level("q", Year, lev_b)['q'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
    
    # trapezoidal integration, must average the integrand at both level boundaries
    # Q units: kg/kg, U/V units: m/s, dp units: Pa, so IVT units: km/m/s
    IVT_u += (Q_a * U_a + Q_b * U_b) / 2 * dp / 9.81 
    IVT_v += (Q_a * V_a + Q_b * V_b) / 2 * dp / 9.81 

# compute IVT magnitude from zonal and meridional components
IVT = np.sqrt(IVT_u**2 + IVT_v**2)

# coord arrays pulled from last loaded level for use in plotting
lats = U_a.latitude
lons = U_a.longitude

print(f"IVT calculated, Peak magnitude: {float(IVT.max()):.1f} kg/m/s")

# load MSLP and convert from Pa to hPa for plotting
# show steering and enhancing moisture transport
dsMSLP = get_era5_sfc_ds("slp", Year)
MSLP = dsMSLP['msl'].sel(time=dateTime, latitude=slice(latN+5, latS-5), longitude=slice(lonW-5, lonE+5))
MSLP = MSLP / 100  # Pa to hPa

print(f"MSLP loaded, Range: {float(MSLP.min()):.1f}–{float(MSLP.max()):.1f} hPa")

## 4.) IVT and MSLP Plot

Three overlapping layers are plotted:

1. **IVT filled contours** (GnBu): moisture transport intensity; 250 kg/m/s (standard atmospheric river threshold)
2. **IVT vectors** (quiver): direction and relative magnitude of moisture flux
3. **MSLP contours** (red): surface pressure systems at 4 hPa intervals (960–1040 hPa)

In [ ]:
# set up map projections
proj_map = ccrs.LambertConformal(central_longitude=-100, central_latitude=30)
proj_data = ccrs.PlateCarree()  # Lat-lon data projection

# IVT contour intervals (kg/m/s)
IVTcintervals = np.arange(250, 1250, 250) # 250kg/m/s threshold usually used for atmospheric reivers analysis

# PLOT
fig, ax = plt.subplots(subplot_kw={'projection': proj_map}, figsize=(12, 8))

# set extent
#ax.set_extent([-127, -67, 23, 50], crs=ccrs.PlateCarree()) # for the inner 48 states
ax.set_extent([-160, -67, 25, 60], crs=ccrs.PlateCarree()) # to include alaska
# ax.set_extent([-105, -80, 35, 50], crs=ccrs.PlateCarree()) # midwest

# political/geographic boundaries
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle='--', linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')

# lon/lat gridlines
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
gridlines.top_labels   = True
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 12, 'color': 'black'}
gridlines.ylabel_style = {'size': 15, 'color': 'black'}

# 1. IVT filled countours
# show moisture transport intensity
ivt_fill = ax.contourf(
    lons, lats, IVT, 
    levels=IVTcintervals, 
    cmap='GnBu', alpha=0.85, 
    transform=proj_data, 
    extend='max', zorder=1
)

# 2. IVT contour lines and labels
ivt_contour = ax.contour(
    lons, lats, IVT,
    levels=IVTcintervals,
    colors='black', linewidths=0.8,
    linestyles='solid',
    transform=proj_data, zorder=2
)
ax.clabel(ivt_contour, inline=True, fmt='%d', fontsize=10, colors='black', inline_spacing=3)

# 3. IVT vectors
# show direction and relative magnitude of moisture transport
skip = 8 # to avoid crowding the map
q = ax.quiver(
    lons[::skip], lats[::skip],
    IVT_u.values[::skip, ::skip], IVT_v.values[::skip, ::skip],
    transform=proj_data,
    scale=10000, # IVT magnitudes are much larger than wind, adjust as needed
    color='black',
    width=0.002, # arrow shaft thickness relative to plot width
    alpha=0.8,   
    zorder = 3
)

# 4. MSLP countours and labels
# highlights pressure systems associated with moisture transport
contour_mslp = ax.contour(
    lons, lats, MSLP, 
    levels=np.arange(960, 1040, 4),
    colors='red', linewidths=0.8, 
    linestyles='solid', 
    transform=proj_data, zorder=3
)
ax.clabel(contour_mslp, inline=True, fmt='%d', fontsize=10, colors='red', inline_spacing=3)

# IVT contour lines
contour = ax.contour(
    lons, lats, IVT, 
    levels=IVTcintervals, 
    colors='black', linewidths=.8, 
    linestyles='solid', 
    transform=proj_data, zorder=2
)

# colorbar for IVT
cbar = plt.colorbar(ivt_fill, ax=ax, orientation='vertical',
                    pad=0.04, aspect=30, shrink=0.8, extendrect=True)
cbar.set_label('IVT (kg/m/s)', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# title
title =  "ERA-5 Integrated Vapor Transport (IVT) and Surface Pressure (MSLP)"
subtitle = f"Valid at: {timeStr}"
plt.title(f"{title}\n{subtitle}", fontsize=16, fontweight='bold', loc='center')

# save and show
plt.tight_layout()

plot_path = plot_dir / f"IVT_MSLP_{timeStr}.png"
plt.savefig(plot_path, dpi=100)
print(f"Plot saved at: {plot_path}")

plt.show()